---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-34: LangChain <b>Document Loaders</b> </h1>

# Learning agenda of this notebook

> **The idea of this lecture:** an LLM knows nothing about *your* files: your PDFs, spreadsheets, web pages or database. Before a RAG app can search them, they must be read and turned into one common shape. A **document loader** reads one kind of source and returns a list of LangChain **`Document`** objects: the text, plus facts about where it came from.

1. **Where Document Loaders Fit**
2. **The `Document` Object**
    - Example 1: Create a `Document` by hand
3. **Loading Files**
    - Example 2: `TextLoader`: a `.txt` file
    - Example 3: `Docx2txtLoader`: a Word file
    - Example 4: `PyPDFLoader`: a PDF file, page by page
    - Example 5: `CSVLoader` and `UnstructuredCSVLoader`: a CSV file
    - Example 6: `UnstructuredExcelLoader`: an Excel file
    - Example 7: `JSONLoader`: a JSON file
4. **Loading a Web Page**
    - Example 8: `WebBaseLoader`
5. **Loading Many Files, and Big Files**
    - Example 9: `DirectoryLoader`, with `load()` and `lazy_load()`
6. **Loading from a Database**
    - Example 10: `SQLDatabaseLoader`
7. **Writing Your Own Loader**
    - Example 11: A custom loader with `BaseLoader`
8. **Summary**

> **Note:** this lecture calls no LLM at all, so it needs no API key. Example 8 needs an internet connection.

# <span style='background :lightgreen' >1. Where Document Loaders Fit</span>

A RAG app first prepares your documents, once, and then answers questions from them. **Loading** is the very first step:

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **Load (this lecture):** read each source and return a list of `Document` objects.
- **Split (Lec-35):** cut large Documents into small chunks.
- **Embed and store (Lec-36):** turn each chunk into a vector and save it in a vector store.
- **Retrieve (Lec-37) and generate (Lec-38):** find the chunks that match a question and give them to the model.

**Every loader works the same way:** create it with the source (a path, a URL, a query), then call `.load()`:
```python
loader = PyPDFLoader("../data/attention.pdf")   # 1. which source to read
docs = loader.load()                            # 2. read it: a list of Document objects
```

**Where loaders live:**
- Most loaders are in the `langchain-community` package: `from langchain_community.document_loaders import ...`. They are written by the community; each one uses a normal Python library under the hood (`pypdf`, `docx2txt`, `openpyxl`, `jq`, `beautifulsoup4`, ...).
- Some companies publish their own packages, for example `langchain-docling` and `langchain-unstructured`.
- See the list of all loaders: https://docs.langchain.com/oss/python/integrations/document_loaders

> **Note:** all the libraries used in this lecture are already in the course environment (`uv sync`). In your own project you add them with `uv add`, for example `uv add langchain-community pypdf`.

In [1]:
import os                                                    # to set an environment variable

# WebBaseLoader (Example 8) warns if no USER_AGENT is set: websites use it to see who is reading the page
os.environ["USER_AGENT"] = "genai-course-lec34"

# <span style='background :lightgreen' >2. The <code>Document</code> Object</span>

Every loader returns the same thing: a list of `Document` objects. A Document has two main parts:

<div style="text-align:center;">
    <img src="../images/langchain-new-19.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- **`page_content`** (a `str`): the text. This is what gets split, embedded, searched and finally pasted into the prompt.
- **`metadata`** (a `dict`): facts about the text, such as the file name and page number. The model does not search it, but your app uses it to show sources (*"see page 3 of attention.pdf"*) and to filter searches (*"only pages 1 to 5"*).
- **`id`** (optional): a unique name. Loaders leave it as `None`; a vector store sets it (Lec-36).

## <span style='color:#7e22ce'>Example 1: Create a <code>Document</code> by hand</span>

A loader does nothing magical: it reads a file and builds objects like this one.

In [2]:
from langchain_core.documents import Document              # the class that every loader returns

doc = Document(
    page_content="LangChain document loaders read files and turn them into Document objects.",  # the text
    metadata={                                              # facts about the text: any keys you like
        "source": "my_notes.txt",                           # where the text came from
        "author": "Arif Butt",                              # who wrote it
        "topic": "RAG",                                     # a label we could filter on later
    },
)

print(type(doc))                                            # a Document object
print("page_content:", doc.page_content)                    # the text, a str
print("metadata    :", doc.metadata)                        # the facts, a dict
print("source      :", doc.metadata["source"])              # read one key like any dict
print("id          :", doc.id)                              # None: nobody has given it an id yet

<class 'langchain_core.documents.base.Document'>
page_content: LangChain document loaders read files and turn them into Document objects.
metadata    : {'source': 'my_notes.txt', 'author': 'Arif Butt', 'topic': 'RAG'}
source      : my_notes.txt
id          : None


# <span style='background :lightgreen' >3. Loading Files</span>

All loaders share `.load()`, but they do **not** all cut the source the same way. A text file becomes one Document, a PDF one Document per page, a CSV one Document per row. This figure shows what each loader in this lecture returns, with the real counts from our files:

<div style="text-align:center;">
    <img src="../images/langchain-new-20.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 2: <code>TextLoader</code>: a <code>.txt</code> file</span>

`TextLoader` reads a plain-text file (`.txt`, `.md`, `.log`, ...) and returns **one Document for the whole file**.

In [3]:
from langchain_community.document_loaders import TextLoader   # loader for plain-text files

loader = TextLoader("../data/arif_bio.txt", encoding="utf-8")   # which file, and how its bytes are decoded
docs = loader.load()                                            # read it: always a list

print(type(docs), "with", len(docs), "Document")                # a list with 1 Document
print("metadata    :", docs[0].metadata)                        # TextLoader only records the path
print("characters  :", len(docs[0].page_content))               # the whole file is in one Document
print("page_content:", docs[0].page_content[:200], "...")      # the first 200 characters

<class 'list'> with 1 Document
metadata    : {'source': '../data/arif_bio.txt'}
characters  : 1692
page_content: Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and  ...


## <span style='color:#7e22ce'>Example 3: <code>Docx2txtLoader</code>: a Word file</span>

`Docx2txtLoader` reads a Word file (`.docx`) with the `docx2txt` library. It keeps the **text only**: no images, fonts or tables layout. One Document for the whole file.

In [4]:
from langchain_community.document_loaders import Docx2txtLoader   # loader for .docx files

loader = Docx2txtLoader("../data/proposal.docx")    # the Word file
docs = loader.load()                                # read it

print("Documents   :", len(docs))                   # one Document for the whole file
print("metadata    :", docs[0].metadata)            # only the path
print("page_content:")                              # the text, with the formatting removed
print(docs[0].page_content[:300], "...")            # the first 300 characters

Documents   : 1
metadata    : {'source': '../data/proposal.docx'}
page_content:
Project Proposal: RAG Implementation

Executive Summary

This proposal outlines the implementation of a Retrieval-Augmented Generation system for our organization.

Objectives

Key objectives include:

• Improve information retrieval accuracy

• Reduce response time for customer queries

• Integrate ...


## <span style='color:#7e22ce'>Example 4: <code>PyPDFLoader</code>: a PDF file, page by page</span>

`PyPDFLoader` reads a PDF with the `pypdf` library and returns **one Document per page**. Each page keeps its page number in the metadata, so an answer can say *"see page 3"*. With `mode="single"` it returns the whole PDF as one Document instead.

In [5]:
from langchain_community.document_loaders import PyPDFLoader   # loader for PDF files

loader = PyPDFLoader("../data/attention.pdf")      # the paper "Attention Is All You Need"
docs = loader.load()                               # default mode="page": one Document per page

print("Documents:", len(docs))                     # one per page
page3 = docs[2]                                    # the third page (lists start at 0)

# the PDF gives many metadata keys; these four are the useful ones
for key in ["source", "page", "page_label", "total_pages"]:
    print(f"  {key:12}: {page3.metadata[key]}")    # page counts from 0, page_label is the printed number
print("  ... and", len(page3.metadata) - 4, "more keys copied from the PDF file")

print("\npage_content of page 3:")
print(page3.page_content[:250], "...")              # the text pypdf extracted from that page

Documents: 15
  source      : ../data/attention.pdf
  page        : 2
  page_label  : 3
  total_pages : 15
  ... and 10 more keys copied from the PDF file

page_content of page 3:
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
re ...


In [6]:
# the same PDF as ONE Document
single = PyPDFLoader("../data/attention.pdf", mode="single").load()   # mode="single": no page split

print("Documents :", len(single))                                       # just one
print("characters:", len(single[0].page_content))                       # the text of all 15 pages
print("has 'page' in metadata?", "page" in single[0].metadata)          # no page number any more

Documents : 1
characters: 39643
has 'page' in metadata? False


**`page` vs `page_label`:** `page` counts from 0 (Python style); `page_label` is the number printed on the page. Use `page_label` when you show a source to a user.

**Which PDF loader?** `pypdf` reads the text layer of a PDF. It cannot read **scanned** pages (they are pictures) and it may mix up tables and two-column layouts. Other loaders exist for those cases:

| Your PDF | Loader | Package |
|---|---|---|
| Normal, text-based PDF | `PyPDFLoader` | `pypdf` |
| Faster, can also extract images | `PyMuPDFLoader` | `pymupdf` |
| Tables | `PDFPlumberLoader` | `pdfplumber` |
| Output as Markdown (headings, tables) for LLMs | `PyMuPDF4LLMLoader` | `langchain-pymupdf4llm` |
| Scanned pages, complex layouts | `DoclingLoader`, `UnstructuredLoader` | `langchain-docling`, `langchain-unstructured` |

> **Note:** Docling and Unstructured can read scanned pages with OCR (Optical Character Recognition), but they download large models. They are not installed in the course environment.

## <span style='color:#7e22ce'>Example 5: <code>CSVLoader</code> and <code>UnstructuredCSVLoader</code>: a CSV file</span>

- `CSVLoader` returns **one Document per row**. Each row is written as `column: value` lines, so the model sees the column names next to the values.
- `source_column` puts the value of one column into `metadata["source"]` instead of the file name.
- `UnstructuredCSVLoader` returns the **whole table as one Document**.

In [7]:
from langchain_community.document_loaders import CSVLoader   # one Document per row

loader = CSVLoader(
    file_path="../data/products.csv",   # 5 products, 5 columns
    encoding="utf-8",                   # how the file is decoded
    source_column="Product",            # metadata["source"] = the product name instead of the file name
)
docs = loader.load()                    # read it

print("Documents:", len(docs))          # one per row (the header row is not a Document)
print("\npage_content of the first row:")
print(docs[0].page_content)             # 'column: value' lines
print("\nmetadata of each row:")
for d in docs:
    print(" ", d.metadata)              # source = product name, row = row number from 0

Documents: 5

page_content of the first row:
Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD

metadata of each row:
  {'source': 'Laptop', 'row': 0}
  {'source': 'Mouse', 'row': 1}
  {'source': 'Keyboard', 'row': 2}
  {'source': 'Monitor', 'row': 3}
  {'source': 'Webcam', 'row': 4}


In [8]:
from langchain_community.document_loaders import UnstructuredCSVLoader   # the whole table as one Document

docs = UnstructuredCSVLoader("../data/products.csv").load()   # default mode="single"

print("Documents:", len(docs))                                 # one for the whole file
print(docs[0].page_content[:200], "...")                       # all cells in one string: column names are no longer next to each value

Documents: 1
Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keybo ...


For a table, `CSVLoader` is usually the better choice: each row is a small Document with its column names, so a question such as *"What does the Monitor cost?"* can find exactly the right row.

## <span style='color:#7e22ce'>Example 6: <code>UnstructuredExcelLoader</code>: an Excel file</span>

`UnstructuredExcelLoader` reads `.xlsx` and `.xls` files with the `unstructured` library.
- `mode="single"` (default): the whole workbook as one Document.
- `mode="elements"`: one Document per table. Our file has **two sheets**, each with one table, so we get two Documents.

In [9]:
from langchain_community.document_loaders import UnstructuredExcelLoader   # loader for Excel files

loader = UnstructuredExcelLoader("../data/inventory.xlsx", mode="elements")  # one Document per table
docs = loader.load()                                                       # read it

print("Documents:", len(docs))                                             # 2: the file has 2 sheets
for d in docs:
    print("\nsheet:", d.metadata["page_name"])                             # the sheet name is in the metadata
    print("text :", d.page_content[:120], "...")                           # the cells as plain text
print("\nthe table as HTML is kept too:", docs[1].metadata["text_as_html"][:100], "...")

Documents: 2

sheet: Products
text : Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SS ...

sheet: Summary
text : Category Total_Items Total_Value Electronics 3 1389.97 Accessories 2 109.98 ...

the table as HTML is kept too: <table><tr><td>Category</td><td>Total_Items</td><td>Total_Value</td></tr><tr><td>Electronics</td><td ...


## <span style='color:#7e22ce'>Example 7: <code>JSONLoader</code>: a JSON file</span>

A JSON file has no pages or rows, so you tell `JSONLoader` **which part** becomes a Document, with a small `jq` query:
- `jq_schema=".employees[]"` means *"each item of the `employees` list"*: one Document per employee.
- `content_key="role"` keeps only that field as the text.
- `metadata_func` copies other fields into the metadata.

In [10]:
import json                                                   # to look at the raw file first
from langchain_community.document_loaders import JSONLoader  # loader for JSON files (uses the jq library)

with open("../data/company_data.json") as f:                 # the raw file: a company with a list of employees
    print("keys of the file:", list(json.load(f).keys()))

# 1. each employee becomes one Document, with the whole employee object as its text
loader = JSONLoader(
    file_path="../data/company_data.json",
    jq_schema=".employees[]",     # jq query: each item of the employees list
    text_content=False,           # the items are objects, not strings: store them as JSON text
)
docs = loader.load()
print("\nDocuments:", len(docs))                              # 2 employees, 2 Documents
print("page_content:", docs[0].page_content)                  # the first employee as JSON text
print("seq_num     :", docs[0].metadata["seq_num"])          # its position, counted from 1

keys of the file: ['company', 'employees', 'departments']

Documents: 2
page_content: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}
seq_num     : 1


In [11]:
# 2. only the "role" field as text, and the employee's name in the metadata
def add_name(record: dict, metadata: dict) -> dict:
    """Called by JSONLoader for each item: record is the employee, metadata is what the loader made."""
    metadata["name"] = record["name"]          # copy one field of the item into the metadata
    return metadata                            # JSONLoader stores what we return

loader = JSONLoader(
    file_path="../data/company_data.json",
    jq_schema=".employees[]",                  # each employee
    content_key="role",                        # the text is only the role
    metadata_func=add_name,                    # and the name goes into the metadata
)
for d in loader.load():
    print(f"{d.metadata['name']:11} -> page_content = {d.page_content!r}")

John Doe    -> page_content = 'Software Engineer'
Jane Smith  -> page_content = 'Data Scientist'


> **Note:** `JSONLoader` stores the full path of the file in `metadata["source"]` (for example `/Users/you/.../company_data.json`). That is why we print other keys here.

# <span style='background :lightgreen' >4. Loading a Web Page</span>

## <span style='color:#7e22ce'>Example 8: <code>WebBaseLoader</code></span>

`WebBaseLoader` downloads a page and extracts its text with **BeautifulSoup**, the HTML library of Python. One Document per URL.

A web page contains much more than its article: menus, sidebars, links and footers. That text is noise in a RAG app. `bs_kwargs` with a `SoupStrainer` tells BeautifulSoup to keep only one part of the page. On Wikipedia, the article is the HTML element with `id="mw-content-text"`.

In [12]:
import bs4                                                           # BeautifulSoup: reads HTML
from langchain_community.document_loaders import WebBaseLoader      # loader for web pages

url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"

# 1. the whole page
page = WebBaseLoader(url).load()[0]                                  # one URL, one Document
print("whole page :", len(page.page_content), "characters")
print("metadata   :", page.metadata)                                 # the page title is added for us
print("starts with:", repr(page.page_content[:120]))                 # menus and empty lines first

# 2. only the article
only_article = bs4.SoupStrainer(id="mw-content-text")                # keep only this HTML element
article = WebBaseLoader(url, bs_kwargs={"parse_only": only_article}).load()[0]
print("\narticle    :", len(article.page_content), "characters")
print("starts with:", repr(article.page_content[:200]))             # the article text itself

whole page : 25031 characters
metadata   : {'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'title': 'Retrieval-augmented generation - Wikipedia', 'language': 'en'}
starts with: '\n\n\n\nRetrieval-augmented generation - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain me'



article    : 22086 characters
starts with: 'Type of information retrieval using LLMs\nRetrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data so'


> **Note:** `WebBaseLoader` reads the HTML the server sends. Pages that build their content with JavaScript come back almost empty; for those there are browser-based loaders such as `PlaywrightURLLoader`. Always check that a website allows its pages to be copied before loading many of them.

# <span style='background :lightgreen' >5. Loading Many Files, and Big Files</span>

## <span style='color:#7e22ce'>Example 9: <code>DirectoryLoader</code>, with <code>load()</code> and <code>lazy_load()</code></span>

`DirectoryLoader` loads every file of a folder that matches a **glob** pattern, using the loader you give it in `loader_cls`, one Document per file.

| Glob pattern | Matches |
|---|---|
| `*.txt` | the `.txt` files in the folder itself |
| `**/*.txt` | the `.txt` files in the folder and all its sub-folders |
| `**/*.pdf` | all PDF files, in all sub-folders |

> **Note:** `loader_cls` is **one** loader class, so pick a glob that matches one file type. For a folder with PDFs and text files, use two `DirectoryLoader`s and add the two lists. Without `loader_cls`, `DirectoryLoader` uses `UnstructuredFileLoader`, which guesses the type of each file.

**`load()` or `lazy_load()`?** Every loader has both.

<div style="text-align:center;">
    <img src="../images/langchain-new-21.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

In [13]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "../data/",                           # the folder
    glob="*.txt",                         # which files: every .txt file in the folder
    loader_cls=TextLoader,                # the loader used for each file
    loader_kwargs={"encoding": "utf-8"},  # extra arguments passed to TextLoader
)

# load(): read ALL files now, and get a list
docs = loader.load()
print("load() returned a", type(docs).__name__, "with", len(docs), "Documents")
for d in docs:
    print(f"  {d.metadata['source']:28} {len(d.page_content):5} characters")

load() returned a list with 6 Documents
  ../data/cricket.txt           4591 characters
  ../data/names.txt             1176 characters
  ../data/python_intro.txt       489 characters
  ../data/machine_learning.txt   575 characters
  ../data/proposal.txt           484 characters
  ../data/arif_bio.txt          1692 characters


In [14]:
# lazy_load(): a generator. Nothing is read until the loop asks for the next Document
docs_gen = loader.lazy_load()
print("lazy_load() returned a", type(docs_gen).__name__)    # generator: no file read yet

for d in docs_gen:                                          # each step reads ONE file
    words = len(d.page_content.split())                     # "process" it: here we just count words
    print(f"  read {d.metadata['source']:28} -> {words:4} words, then forget it")

lazy_load() returned a generator
  read ../data/cricket.txt          ->  714 words, then forget it
  read ../data/names.txt            ->  173 words, then forget it
  read ../data/python_intro.txt     ->   71 words, then forget it
  read ../data/machine_learning.txt ->   77 words, then forget it
  read ../data/proposal.txt         ->   66 words, then forget it
  read ../data/arif_bio.txt         ->  228 words, then forget it


# <span style='background :lightgreen' >6. Loading from a Database</span>

## <span style='color:#7e22ce'>Example 10: <code>SQLDatabaseLoader</code></span>

Much company knowledge sits in SQL databases. `SQLDatabaseLoader` runs a SQL query and returns **one Document per result row**.

First we create a small SQLite database with two related tables. SQLite is a database in a single file: no server, no users and passwords, just a path (we used it for memory in Lec-32).
- **`programs`**: `program_id`, `program_name`, `university`, `duration_years`
- **`students`**: `id`, `name`, `age`, `program_id`, `enrollment_year`; `program_id` points to a row of `programs` (a foreign key)

In [15]:
import sqlite3                                       # SQLite comes with Python
import os                                            # to delete an old copy of the file

DB_PATH = "../data/temp_students.db"                 # the database file this notebook creates
if os.path.exists(DB_PATH):                          # start fresh on every run
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)                      # creates the file
conn.execute("""CREATE TABLE programs (
    program_id     INTEGER PRIMARY KEY,              -- unique id of the program
    program_name   TEXT NOT NULL,                    -- name of the degree
    university     TEXT NOT NULL,                    -- who offers it
    duration_years INTEGER)""")
conn.execute("""CREATE TABLE students (
    id              INTEGER PRIMARY KEY,             -- unique id of the student
    name            TEXT NOT NULL,
    age             INTEGER,
    program_id      INTEGER REFERENCES programs(program_id),  -- foreign key: which program
    enrollment_year INTEGER)""")

conn.executemany("INSERT INTO programs (program_name, university, duration_years) VALUES (?, ?, ?)", [
    ("Computer Science", "MIT", 4), ("Data Science", "Stanford University", 2),
    ("Business Administration", "Harvard University", 2), ("Electrical Engineering", "MIT", 4),
    ("Medicine", "Johns Hopkins University", 6)])
conn.executemany("INSERT INTO students (name, age, program_id, enrollment_year) VALUES (?, ?, ?, ?)", [
    ("Alisha", 21, 1, 2025), ("Abeeha", 22, 2, 2022), ("Khalood", 22, 2, 2022), ("Jamil", 30, 3, 2024),
    ("Rauf", 42, 4, 2023), ("Haider", 36, 5, 2025), ("Mujahid", 25, 1, 2023)])
conn.commit()                                        # save the rows to the file

for table in ["programs", "students"]:               # check what we stored
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:9}: {n} rows")
conn.close()

programs : 5 rows
students : 7 rows


You can look at the same database in a terminal with the `sqlite3` program:
```bash
$ cd genai-course/data/
$ sqlite3 temp_students.db
sqlite> .tables
sqlite> .schema students
sqlite> .headers on
sqlite> .mode table
sqlite> SELECT * FROM students;
sqlite> SELECT s.name, p.program_name, p.university FROM students s JOIN programs p ON s.program_id = p.program_id;
sqlite> .quit
```

In [16]:
from langchain_community.utilities import SQLDatabase                 # a connection to any SQL database
from langchain_community.document_loaders import SQLDatabaseLoader    # one Document per result row

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")     # SQLAlchemy URI: sqlite:/// + the file path
print("tables:", db.get_usable_table_names())

# a JOIN: each student with the program and university, as one row
QUERY = """
SELECT s.name, s.age, p.program_name, p.university, s.enrollment_year
FROM students s JOIN programs p ON s.program_id = p.program_id
ORDER BY s.name
"""

docs = SQLDatabaseLoader(QUERY, db).load()            # run the query, one Document per row
print("Documents:", len(docs))
print("\npage_content of the first row:")
print(docs[0].page_content)                           # 'column: value' lines, like CSVLoader
print("\nmetadata:", docs[0].metadata)                # empty by default

tables: ['programs', 'students']
Documents: 7

page_content of the first row:
name: Abeeha
age: 22
program_name: Data Science
university: Stanford University
enrollment_year: 2022

metadata: {}


The default text works, but a **sentence** reads better for a model, and the metadata is empty. Two small functions fix both: `page_content_mapper` builds the text of each row, `metadata_mapper` builds its metadata. Each one receives the row as a dict.

In [17]:
def row_to_sentence(row: dict) -> str:
    """The text of one Document: the row as a sentence."""
    return (f"{row['name']} (age {row['age']}) studies {row['program_name']} "
            f"at {row['university']}, enrolled in {row['enrollment_year']}.")

def row_to_metadata(row: dict) -> dict:
    """The metadata of one Document: fields we may want to filter on later."""
    return {"source": DB_PATH, "student": row["name"], "university": row["university"]}

loader = SQLDatabaseLoader(QUERY, db,
                           page_content_mapper=row_to_sentence,   # how to write the text
                           metadata_mapper=row_to_metadata)       # how to fill the metadata
for d in loader.load():
    print(d.page_content, "|", d.metadata["university"])

Abeeha (age 22) studies Data Science at Stanford University, enrolled in 2022. | Stanford University
Alisha (age 21) studies Computer Science at MIT, enrolled in 2025. | MIT
Haider (age 36) studies Medicine at Johns Hopkins University, enrolled in 2025. | Johns Hopkins University
Jamil (age 30) studies Business Administration at Harvard University, enrolled in 2024. | Harvard University
Khalood (age 22) studies Data Science at Stanford University, enrolled in 2022. | Stanford University
Mujahid (age 25) studies Computer Science at MIT, enrolled in 2023. | MIT
Rauf (age 42) studies Electrical Engineering at MIT, enrolled in 2023. | MIT


# <span style='background :lightgreen' >7. Writing Your Own Loader</span>

## <span style='color:#7e22ce'>Example 11: A custom loader with <code>BaseLoader</code></span>

When no loader fits your data, write one. Every LangChain loader is a subclass of **`BaseLoader`**, and you only need to write **one method**: `lazy_load()`, a generator that `yield`s one Document at a time. `BaseLoader` then gives you `load()` for free (it is just `list(self.lazy_load())`, see the figure in Section 5).

Our file `drarif.jsonl` has one JSON object per line: a question about Dr. Arif Butt and its answer, as a `user` and an `assistant` message. We want **one Document per question and answer**.

In [18]:
import json                                                  # each line of the file is one JSON object
from itertools import islice                                 # take the first few items of a generator
from typing import Iterator                                  # type hint for a generator
from langchain_core.document_loaders import BaseLoader       # the parent class of every loader
from langchain_core.documents import Document

class FAQLoader(BaseLoader):
    """Loads a .jsonl file of chat examples: one Document per question and answer."""

    def __init__(self, file_path: str):
        self.file_path = file_path                           # remember the path; read nothing yet

    def lazy_load(self) -> Iterator[Document]:
        """Read the file line by line and yield one Document per line."""
        with open(self.file_path, encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):      # one line = one example
                messages = json.loads(line)["messages"]      # [{"role": "user", ...}, {"role": "assistant", ...}]
                question = messages[0]["content"]            # the user message
                answer = messages[1]["content"]              # the assistant message
                yield Document(                              # hand over ONE Document, then pause here
                    page_content=f"Q: {question}\nA: {answer}",
                    metadata={"source": self.file_path, "line": line_no},
                )

loader = FAQLoader("../data/drarif.jsonl")

for d in islice(loader.lazy_load(), 2):                       # only the first 2: the rest of the file is not read
    print(d.metadata)
    print(d.page_content[:200], "...\n")

print("load() gives all of them:", len(loader.load()), "Documents")   # load() comes from BaseLoader

{'source': '../data/drarif.jsonl', 'line': 1}
Q: Who is Dr. Muhammad Arif Butt?
A: Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He has over 33 ye ...

{'source': '../data/drarif.jsonl', 'line': 2}
Q: Tell me about Dr. Arif Butt
A: Dr. Muhammad Arif Butt is an Assistant Professor at Punjab University's Data Science department. He's a technology entrepreneur and cybersecurity expert with 33+ year ...

load() gives all of them: 1926 Documents


# <span style='background :lightgreen' >8. Summary</span>

| Ex | Source | Loader | One Document = | Library |
|---|---|---|---|---|
| 1 | your own text | `Document(...)` | what you put in it | `langchain-core` |
| 2 | `.txt`, `.md` | `TextLoader` | the whole file | none |
| 3 | `.docx` | `Docx2txtLoader` | the whole file | `docx2txt` |
| 4 | `.pdf` | `PyPDFLoader` | a page (`mode="single"`: the file) | `pypdf` |
| 5 | `.csv` | `CSVLoader` / `UnstructuredCSVLoader` | a row / the whole table | none / `unstructured` |
| 6 | `.xlsx`, `.xls` | `UnstructuredExcelLoader` | the workbook / a table (`mode="elements"`) | `unstructured`, `openpyxl` |
| 7 | `.json` | `JSONLoader` | each match of a `jq` query | `jq` |
| 8 | a web page | `WebBaseLoader` | a URL | `beautifulsoup4` |
| 9 | a folder | `DirectoryLoader` | a file, with the loader you choose | depends on `loader_cls` |
| 10 | a SQL database | `SQLDatabaseLoader` | a row of the query | `sqlalchemy` |
| 11 | anything | your own `BaseLoader` | whatever `lazy_load()` yields | your code |

### Key takeaways
- A **document loader** reads one kind of source and returns a list of **`Document`** objects.
- A Document is **`page_content`** (the text that is searched) plus **`metadata`** (facts used to cite sources and filter searches).
- All loaders share **`.load()`** and **`.lazy_load()`**; they differ in how they cut the source: per file, page, row, sheet, match or line.
- Use **`lazy_load()`** for many or large files: one Document in memory at a time.
- Clean text matters: remove noise early (Example 8), and write rows as readable sentences (Example 10).
- When no loader fits, subclass **`BaseLoader`** and write `lazy_load()`.
- **Next lecture (Lec-35):** **Text splitters**: cutting large Documents into small chunks.

### Check your understanding
1. What are the two main parts of a `Document`, and what is each one used for in a RAG app?
2. Why does `PyPDFLoader` return 15 Documents for `attention.pdf`, and how do you get one instead?
3. What is the difference between `metadata["page"]` and `metadata["page_label"]`?
4. Why is `CSVLoader` often better than `UnstructuredCSVLoader` for answering questions about a table?
5. What does `jq_schema=".employees[]"` mean in Example 7?
6. Why did we use a `SoupStrainer` in Example 8? What would happen without it?
7. When would you use `lazy_load()` instead of `load()`?
8. In Example 10, what did `page_content_mapper` change, and why does it help the model?
9. Which method must a custom loader implement, and where does its `load()` come from?